# 06 — E1 leave-one-generator-out (Akshat Yuvan)

**Question.** Published detectors report near-perfect accuracy because the
test generators are the same ones the model trained on. E1 measures what
happens when a generator is **withheld entirely** from training and appears
only at test time, the only case that matters in deployment.

**Protocol, per fold** (binary real vs synthetic, locked decision 1):
- Train and validate on real clips plus the **four seen generators**, using
  the same 10,697-clip subset and ytid-disjoint splits as M1–M3, minus every
  clip of the withheld generator (it is removed from val too, so checkpoint
  selection never peeks at it).
- **Closed-set test:** test-split real clips + test-split clips of the four
  seen generators (cached, as in the M1–M3 table).
- **Open-set test:** the same 804 test-split real clips + **all 804**
  test-split clips of the withheld generator, with features computed here
  from raw audio by the exact M1/M2 feature code. Only test-split ytids are
  used, so no musical content seen in training reaches the open-set test.
- The gap between closed-set and open-set balanced accuracy is the finding.

**Folds (3 of 5, the declared scope reduction).** Chosen to span both axes
along which generators differ:
- `MusicGen_medium`: autoregressive over EnCodec tokens, native 32 kHz,
  the only non-diffusion generator.
- `stable_audio_open`: latent diffusion, native 44.1 kHz.
- `mustango`: latent diffusion, native 16 kHz, so no resampling history.

`audioldm2` and `musicldm` are left out: both are 16 kHz latent diffusion
with closely related VAE + vocoder stacks, so they add the least new
information.

**Models.** M1, M2, M3. M2 and M3 are retrained under ONE protocol (M3's
warmup + cosine schedule, grad clipping, 20 epochs, best-val checkpoint on
plain accuracy, decision 10). M1 keeps its original protocol, pinned to one
CPU thread so it is bit-reproducible.

**Persistence.** Each (fold, model) writes its own JSON to `/kaggle/working`
the moment it finishes. Results already present in the attached
`fmc-e1-results` dataset are skipped, so a dropped run costs one model-fold,
not the sweep.

In [ ]:
# Environment and inputs.
import os, glob, json, time, math, copy, shutil
import numpy as np, pandas as pd, torch

print(torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'no GPU attached to this kernel'

def find_input(name, marker, required=True):
    """Inputs mount at /kaggle/input/datasets/<user>/<name>/; fall back to a
    search so a layout change gives a clear message."""
    d = f'/kaggle/input/datasets/akshatyuvan/{name}'
    if os.path.exists(f'{d}/{marker}'):
        return d
    hits = glob.glob(f'/kaggle/input/**/{name}/{marker}', recursive=True)
    if not hits:
        assert not required, f'{name}/{marker} not found under /kaggle/input'
        return None
    return os.path.dirname(hits[0])

MELS_DIR = find_input('fmc-m2-mels', 'mels.npy')
M1_DIR   = find_input('fmc-m1-features', 'X.npy')
MAN_DIR  = find_input('fmc-manifest', 'manifest.csv')
RAW_DIR  = find_input('fakemusiccaps-raw', 'mustango')
DONE_DIR = find_input('fmc-e1-results', 'README.md', required=False)
for d in [MELS_DIR, M1_DIR, MAN_DIR, RAW_DIR, DONE_DIR]:
    print(d, '->', sorted(os.listdir(d))[:12] if d else None)

OUT = '/kaggle/working'
SEED = 42
FOLDS  = ['MusicGen_medium', 'stable_audio_open', 'mustango']
MODELS = ['M1', 'M2', 'M3']

def res_name(fold, model):
    return f'e1_{fold}_{model}.json'

def already_done(fold, model):
    """True if this model-fold has a results file from an earlier run. The
    earlier file is copied into this run's output so every version's output
    holds the complete set."""
    f = res_name(fold, model)
    if os.path.exists(f'{OUT}/{f}'):
        return True
    if DONE_DIR and os.path.exists(f'{DONE_DIR}/{f}'):
        shutil.copy(f'{DONE_DIR}/{f}', f'{OUT}/{f}')
        return True
    return False

print({f: {m: already_done(f, m) for m in MODELS} for f in FOLDS})

In [ ]:
# Cached features for the 10,697-clip subset (the same clips M1–M3 used).
sub    = pd.read_csv(f'{MELS_DIR}/subset.csv', dtype={'ytid': str})
M      = np.load(f'{MELS_DIR}/mels.npy', mmap_mode='r')   # 2.74 GB, paged in on demand
labels = np.load(f'{MELS_DIR}/labels.npy')
X1     = np.load(f'{M1_DIR}/X.npy')                       # M1's 86-dim descriptors
sub1   = pd.read_csv(f'{M1_DIR}/subset.csv', dtype={'ytid': str})
man    = pd.read_csv(f'{MAN_DIR}/manifest.csv', dtype={'ytid': str})

assert M.shape == (10697, 128, 1001) and X1.shape == (10697, 86)
# M2's subset.csv was built from M1's, so the rows must line up one-to-one.
# Then one row index addresses both the mel and the descriptor of a clip.
for c in ['ytid', 'generator', 'label', 'split']:
    assert (sub[c].values == sub1[c].values).all(), f'M1/M2 subsets disagree on {c}'
assert (sub.label.values == labels).all()
assert (man.groupby('ytid').split.nunique() == 1).all()
assert (sub.ytid.map(man.groupby('ytid').split.first()) == sub.split).all()
print(sub.groupby(['split', 'generator']).size().unstack())

In [ ]:
# Feature code for the withheld generator's open-set test clips. Both functions
# are copied verbatim from the M2 (mel) and M1 (descriptor) notebooks, so the
# open-set clips are processed exactly like the cached ones.
import soundfile as sf, librosa
from joblib import Parallel, delayed

SR, N_MELS, N_FFT, HOP = 16000, 128, 1024, 160
N_SAMPLES = 160000                     # exactly 10.0 s; removes duration as a cue
N_FRAMES  = N_SAMPLES // HOP + 1       # 1001

def mel(path):                         # M2 notebook, cell 2
    y, _ = sf.read(path, dtype='float32')
    y = y[:N_SAMPLES] if len(y) >= N_SAMPLES else np.pad(y, (0, N_SAMPLES - len(y)))
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT,
                                       hop_length=HOP, n_mels=N_MELS)
    S = librosa.power_to_db(S, ref=np.max)
    return S[:, :N_FRAMES].astype(np.float16)

def feats(path):                       # M1 notebook, cell 6
    y, sr = sf.read(path, dtype='float32')
    y = y[:N_SAMPLES] if len(y) >= N_SAMPLES else np.pad(y, (0, N_SAMPLES - len(y)))
    S = np.abs(librosa.stft(y, n_fft=2048, hop_length=512))
    blocks = [
        librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20),
        librosa.feature.chroma_stft(S=S, sr=sr),
        librosa.feature.spectral_contrast(S=S, sr=sr),
        librosa.feature.spectral_centroid(S=S, sr=sr),
        librosa.feature.spectral_rolloff(S=S, sr=sr),
        librosa.feature.spectral_bandwidth(S=S, sr=sr),
        librosa.feature.zero_crossing_rate(y),
    ]
    return np.concatenate([np.concatenate([b.mean(1), b.std(1)]) for b in blocks])

def open_set_features(fold):
    """Mel + descriptor for all 804 test-split clips of the withheld generator.
    Before trusting them, recompute 16 clips that are also in the cache and
    compare: if the Kaggle image's librosa changed since October, the open-set
    clips would be processed differently from everything the models saw."""
    rows = man[(man.split == 'test') & (man.generator == fold)].sort_values('ytid')
    assert len(rows) == 804 and rows.path.map(os.path.exists).all()

    cached = np.where(((sub.split == 'test') & (sub.generator == fold)).values)[0][:16]
    paths = man.set_index(['ytid', 'generator']).path
    chk_paths = [paths[(sub.ytid[j], fold)] for j in cached]
    m_chk = np.stack([mel(p) for p in chk_paths]).astype(np.float32)
    f_chk = np.stack([feats(p) for p in chk_paths])
    mel_diff  = float(np.abs(m_chk - M[cached].astype(np.float32)).max())
    # Descriptor diff in units of each feature's spread across the cache, so
    # features whose values sit near zero cannot inflate a relative error.
    feat_diff = float((np.abs(f_chk - X1[cached]) / (X1.std(0) + 1e-12)).max())
    print(f'{fold}: cache check on 16 clips | mel max |diff| {mel_diff:.4f} dB | '
          f'descriptor max diff {feat_diff:.2e} feature-SDs')
    # fp16 resolution near -45 dB is ~0.03 dB; anything far above that means
    # the pipeline is not reproducing the cache and the fold must not run.
    assert mel_diff < 0.1 and feat_diff < 1e-3, 'feature pipeline does not reproduce the cache'

    t0 = time.time()
    om = np.stack(Parallel(n_jobs=4)(delayed(mel)(p) for p in rows.path))
    of = np.array(Parallel(n_jobs=4)(delayed(feats)(p) for p in rows.path))
    print(f'{fold}: open-set features {om.shape} {of.shape} in {time.time()-t0:.0f}s, '
          f'NaNs {int(np.isnan(om).sum()) + int(np.isnan(of).sum())}')
    return om, of, {"mel_max_abs_diff_db": round(mel_diff, 5),
                    "descriptor_max_diff_in_sd": float(f'{feat_diff:.3e}'),
                    "n_checked": len(cached)}

In [ ]:
# Models: M1, M2, M3 exactly as in their own notebooks.
import torch.nn as nn

def make_m1(n_in=86):
    return nn.Sequential(
        nn.Linear(n_in, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, 128),  nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(128, 1))

class M2CNN(nn.Module):
    """Four conv blocks, each two 3x3 convs + BN + ReLU, then 2x2 maxpool;
    global average pooling; dropout 0.3; one logit. 1,173,857 params."""
    def __init__(self):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(2))
        self.features = nn.Sequential(blk(1,32), blk(32,64), blk(64,128), blk(128,256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(256,1))
    def forward(self, x): return self.head(self.features(x))

class M3CRNN(nn.Module):
    """Frequency-only pooling conv front-end -> BiLSTM over all 1001 frames ->
    mean over time -> one logit. 1,422,913 params."""
    def __init__(self, n_mels=128, hidden=128):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(kernel_size=(2, 1)))
        self.features = nn.Sequential(blk(1, 32), blk(32, 64), blk(64, 128), blk(128, 128))
        self.rnn = nn.LSTM(input_size=128 * (n_mels // 16), hidden_size=hidden,
                           batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * hidden, 1))
    def forward(self, x):
        z = self.features(x)
        b, c, f, t = z.shape
        seq, _ = self.rnn(z.permute(0, 3, 1, 2).reshape(b, t, c * f))
        return self.head(seq.mean(dim=1))

assert sum(p.numel() for p in M2CNN().parameters()) == 1173857
assert sum(p.numel() for p in M3CRNN().parameters()) == 1422913

In [ ]:
# Shared helpers: metric code, datasets, the deep-model training loop.
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)

def metrics(true, score):
    """Line-for-line the M1/M2/M3 metric code."""
    pred = (score > 0.5).astype(int)
    fpr, tpr, _ = roc_curve(true, score)
    eer = float(fpr[np.nanargmin(np.abs((1 - tpr) - fpr))])
    return {"balanced_accuracy": float(balanced_accuracy_score(true, pred)),
            "precision": float(precision_score(true, pred)),
            "recall":    float(recall_score(true, pred)),
            "f1":        float(f1_score(true, pred)),
            "auc":       float(roc_auc_score(true, score)),
            "eer":       eer}

def r4(d):
    return {k: round(v, 4) for k, v in d.items()}

class ArrDS(Dataset):
    """Normalised (1, 128, 1001) mels from any array (the memory-mapped cache
    or the freshly computed open-set array), selected by row index."""
    def __init__(self, arr, rows, ys, mean, std):
        self.arr, self.rows, self.ys, self.mean, self.std = arr, rows, ys, mean, std
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        x = (np.asarray(self.arr[self.rows[i]], dtype=np.float32) - self.mean) / self.std
        return torch.from_numpy(x).unsqueeze(0), torch.tensor(float(self.ys[i]))

def loader(arr, rows, ys, mean, std, shuffle=False, gen=None):
    return DataLoader(ArrDS(arr, rows, ys, mean, std), batch_size=32, shuffle=shuffle,
                      generator=gen, num_workers=2, pin_memory=True)

def predict_deep(model, dl):
    model.eval(); out = []
    with torch.no_grad():
        for xb, _ in dl:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                out.append(torch.sigmoid(model(xb.to('cuda')).squeeze(1).float()).cpu().numpy())
    return np.concatenate(out)

EPOCHS, PEAK_LR, CLIP = 20, 1e-3, 5.0

def train_deep(model, dl_tr, dl_va, y_va, pos_weight, ckpt):
    """The M3 protocol, applied identically to M2 and M3 in every fold:
    Adam 1e-3 / wd 1e-4, warmup 1 epoch + cosine to 0.01x (per step), fp16
    autocast + GradScaler, grad-clip 5, 20 epochs, keep the checkpoint with
    the best PLAIN validation accuracy (decision 10)."""
    steps = EPOCHS * len(dl_tr); warm = len(dl_tr)
    def lr_lambda(s):
        if s < warm: return 0.1 + 0.9 * s / warm
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm)))
    opt = torch.optim.Adam(model.parameters(), lr=PEAK_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    # pos_weight = n_real / n_synth in this fold's train set. Removing one
    # generator leaves ~1.25 real per synthetic clip; weighting the synthetic
    # term rebalances the loss without duplicating clips (decision 5).
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device='cuda'))
    scaler = torch.amp.GradScaler('cuda')
    best, best_ep, best_state, hist, t0 = -1.0, -1, None, [], time.time()
    for ep in range(EPOCHS):
        model.train()
        for xb, yb in dl_tr:
            xb, yb = xb.to('cuda', non_blocking=True), yb.to('cuda', non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                loss = lossf(model(xb).squeeze(1), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(opt); scaler.update(); sched.step()
        s_va = predict_deep(model, dl_va)
        va = float(((s_va > 0.5).astype(int) == y_va).mean())
        hist.append({"epoch": ep, "val_acc": round(va, 4), "elapsed_s": round(time.time() - t0, 1)})
        print(f'    epoch {ep:2d}  val_acc {va:.4f}  ({time.time()-t0:.0f}s)', flush=True)
        if va > best:
            best, best_ep, best_state = va, ep, copy.deepcopy(model.state_dict())
            torch.save(best_state, ckpt)
    model.load_state_dict(best_state)
    return best, best_ep, time.time() - t0, hist

In [ ]:
# The sweep. Each model-fold writes its JSON the moment it finishes.
for fold in FOLDS:
    if all(already_done(fold, m) for m in MODELS):
        print(f'== {fold}: all models already done, skipping'); continue
    print(f'\n========== fold: withhold {fold} ==========')
    seen = sorted(set(sub.generator[sub.label == 1]) - {fold})

    # In-distribution rows: real + the four seen generators. The withheld
    # generator is removed from train AND val.
    keep = (sub.generator != fold).values
    tr = np.where(keep & (sub.split == 'train').values)[0]
    va = np.where(keep & (sub.split == 'val').values)[0]
    te_closed = np.where(keep & (sub.split == 'test').values)[0]
    te_real = np.where(((sub.split == 'test') & (sub.label == 0)).values)[0]
    assert len(te_real) == 804
    y_tr, y_va, y_tc = labels[tr], labels[va], labels[te_closed]
    n_real, n_syn = int((y_tr == 0).sum()), int((y_tr == 1).sum())
    pos_weight = n_real / n_syn
    print(f'train real {n_real} synth {n_syn} (pos_weight {pos_weight:.3f}) | '
          f'val {len(va)} | closed test {len(te_closed)} | seen {seen}')
    assert not (set(sub.ytid[tr]) & set(man.ytid[man.split == 'test']))

    om, of, feat_check = open_set_features(fold)
    y_open = np.r_[np.zeros(len(te_real), int), np.ones(len(om), int)]

    common = {"experiment": "E1_leave_one_generator_out", "fold": fold,
              "withheld_generator": fold, "seen_generators": seen, "seed": SEED,
              "selection_metric": "val_accuracy",
              "n_train_real": n_real, "n_train_synth": n_syn,
              "pos_weight": round(pos_weight, 4), "n_val": int(len(va)),
              "open_set_feature_check": feat_check}

    def finish(model_name, s_closed, s_open, extra):
        """Write one model-fold results file, from variables only (rule 8)."""
        mc, mo = metrics(y_tc, s_closed), metrics(y_open, s_open)
        res = {**common, "model": model_name, **extra,
               "closed_set": {**r4(mc), "n_real": int((y_tc == 0).sum()), "n_synth": int((y_tc == 1).sum())},
               "open_set":   {**r4(mo), "n_real": int(len(te_real)), "n_synth": int(len(om))},
               "delta_bal_acc_open_minus_closed": round(mo["balanced_accuracy"] - mc["balanced_accuracy"], 4)}
        json.dump(res, open(f'{OUT}/{res_name(fold, model_name)}', 'w'), indent=2)
        print(f'  {model_name}: closed {mc["balanced_accuracy"]:.4f} -> open {mo["balanced_accuracy"]:.4f} '
              f'(withheld recall {mo["recall"]:.4f}, open AUC {mo["auc"]:.4f})  -> wrote {res_name(fold, model_name)}')

    # ---- M1: original protocol, CPU, ONE thread so the run is bit-reproducible.
    if not already_done(fold, 'M1'):
        print('  M1 ...')
        torch.manual_seed(SEED); np.random.seed(SEED)
        n_threads = torch.get_num_threads(); torch.set_num_threads(1)
        sc = StandardScaler().fit(X1[tr])                    # train split only (decision 4)
        T = lambda a: torch.tensor(sc.transform(a), dtype=torch.float32)
        Xtr, Xva = T(X1[tr]), T(X1[va])
        ytr_t, yva_t = torch.tensor(y_tr, dtype=torch.float32), torch.tensor(y_va, dtype=torch.float32)
        m1 = make_m1()
        opt = torch.optim.Adam(m1.parameters(), lr=1e-3, weight_decay=1e-4)
        lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))
        best, best_ep, best_state, bad, t0 = 0.0, -1, None, 0, time.time()
        for ep in range(200):
            m1.train(); perm = torch.randperm(len(Xtr))
            for i in range(0, len(perm), 128):
                b = perm[i:i+128]; opt.zero_grad()
                lossf(m1(Xtr[b]).squeeze(1), ytr_t[b]).backward(); opt.step()
            m1.eval()
            with torch.no_grad():
                v = ((m1(Xva).squeeze(1) > 0).float() == yva_t).float().mean().item()
            if v > best:
                best, best_ep, bad = v, ep, 0
                best_state = {k: t.clone() for k, t in m1.state_dict().items()}
            else:
                bad += 1
                if bad >= 15: break
        m1.load_state_dict(best_state); m1.eval(); t1 = time.time() - t0
        torch.save(best_state, f'{OUT}/e1_{fold}_M1.pt')
        with torch.no_grad():
            s_c = torch.sigmoid(m1(T(X1[te_closed])).squeeze(1)).numpy()
            s_o = torch.sigmoid(m1(T(np.r_[X1[te_real], of])).squeeze(1)).numpy()
        torch.set_num_threads(n_threads)
        finish('M1', s_c, s_o, {"device": "cpu (1 thread)", "best_val_acc": round(best, 4),
                                "best_epoch": best_ep, "early_stop_epoch": ep,
                                "train_time_s": round(t1, 1),
                                "config": "Adam 1e-3 wd 1e-4, batch 128, early stop 15, pos_weight"})

    # ---- M2 and M3: one shared deep protocol.
    need = [m for m in ['M2', 'M3'] if not already_done(fold, m)]
    if need:
        # Normalisation from THIS fold's train rows only (decision 4).
        mu = float(M[tr].astype(np.float32).mean()); sd = float(M[tr].astype(np.float32).std())
        print(f'  fold train mel mean/std {mu:.3f} / {sd:.3f}')
        dl_va = loader(M, va, y_va, mu, sd)
        dl_tc = loader(M, te_closed, y_tc, mu, sd)
        dl_real = loader(M, te_real, np.zeros(len(te_real)), mu, sd)
        dl_open = loader(om, np.arange(len(om)), np.ones(len(om)), mu, sd)
    for name in need:
        print(f'  {name} ...')
        torch.manual_seed(SEED); np.random.seed(SEED)
        g = torch.Generator().manual_seed(SEED)
        dl_tr = loader(M, tr, y_tr, mu, sd, shuffle=True, gen=g)
        model = (M2CNN() if name == 'M2' else M3CRNN()).to('cuda')
        best, best_ep, t_s, hist = train_deep(model, dl_tr, dl_va, y_va, pos_weight,
                                              f'{OUT}/e1_{fold}_{name}.pt')
        s_c = predict_deep(model, dl_tc)
        s_o = np.r_[predict_deep(model, dl_real), predict_deep(model, dl_open)]
        finish(name, s_c, s_o, {"device": torch.cuda.get_device_name(0),
                                "best_val_acc": round(best, 4), "best_epoch": best_ep,
                                "train_time_s": round(t_s, 1), "norm_mean": mu, "norm_std": sd,
                                "config": f"Adam {PEAK_LR} wd 1e-4, batch 32, warmup 1 epoch + cosine "
                                          f"to 0.01x per step, {EPOCHS} epochs, clip {CLIP}, "
                                          f"fp16 + GradScaler, pos_weight",
                                "history": hist})
        del model; torch.cuda.empty_cache()
    del om, of

In [ ]:
# Summary: closed-set vs open-set per model and fold, from the results files only.
import matplotlib.pyplot as plt

rows = []
for fold in FOLDS:
    for m in MODELS:
        p = f'{OUT}/{res_name(fold, m)}'
        if not os.path.exists(p):
            print('missing', p); continue
        r = json.load(open(p))
        rows.append({"fold": fold, "model": m,
                     "closed_bal_acc": r["closed_set"]["balanced_accuracy"],
                     "open_bal_acc": r["open_set"]["balanced_accuracy"],
                     "delta": r["delta_bal_acc_open_minus_closed"],
                     "withheld_recall": r["open_set"]["recall"],
                     "closed_recall": r["closed_set"]["recall"],
                     "open_auc": r["open_set"]["auc"], "open_eer": r["open_set"]["eer"]})
summ = pd.DataFrame(rows)
print(summ.to_string(index=False))
summ.to_csv(f'{OUT}/e1_summary.csv', index=False)
if len(summ):
    print('\nmean over folds:')
    print(summ.groupby('model')[['closed_bal_acc', 'open_bal_acc', 'delta', 'withheld_recall', 'open_auc']]
              .mean().round(4).to_string())

    fig, axes = plt.subplots(1, len(FOLDS), figsize=(4 * len(FOLDS), 3.5), sharey=True)
    for ax, fold in zip(np.atleast_1d(axes), FOLDS):
        d = summ[summ.fold == fold]
        x = np.arange(len(d))
        ax.bar(x - 0.2, d.closed_bal_acc, 0.4, label='closed-set')
        ax.bar(x + 0.2, d.open_bal_acc, 0.4, label='open-set (withheld)')
        ax.set_xticks(x); ax.set_xticklabels(d.model); ax.set_title(f'withheld: {fold}')
        ax.axhline(0.5, color='grey', lw=0.8, ls='--'); ax.set_ylim(0.3, 1.0); ax.grid(axis='y', alpha=.3)
    np.atleast_1d(axes)[0].set_ylabel('balanced accuracy'); np.atleast_1d(axes)[0].legend(loc='lower left')
    plt.tight_layout(); plt.savefig(f'{OUT}/e1_closed_vs_open.png', dpi=150); plt.show()
print(sorted(os.listdir(OUT)))